# BirdCLEF 2026 | Ultimate Fusion Notebook
## 3-Mode Rank Fusion × Power Optimization × EoS-4 Vertical Split

| Mode | Strategy | Risk |
|------|----------|------|
| 1 | Optimized Rank Blend (65/35) + Power α=1.2 | Low |
| 2 | EoS-4 Asymmetric Vertical Split + Shifted Cutoff | Medium |
| 3 | Extreme Sharpening (α=1.5) + Heavy ProtoSSM Bias (75/25) | High |


In [ ]:
# ── Cell 0: Install ONNX Runtime + TF 2.20 ────────────────────────────
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

# Try ONNX first (150x faster than TF SavedModel)
ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available ✅")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")


In [ ]:
# ── Cell 1: Mode switch ────────────────────────────────────────────────
MODE = "submit"   # ← change to "train" for local CV
 
assert MODE in {"train", "submit"}
print("MODE =", MODE)


In [ ]:
# ── Cell 2: Imports & config ───────────────────────────────────────────
import os, re, gc, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
 
tf.experimental.numpy.experimental_enable_numpy_behavior()
try: tf.config.set_visible_devices([], "GPU")
except: pass
 
_WALL_START = time.time()
 
BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)
 
SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12          # 12 × 5s = 60s per file
 
CFG = {
    # inference
    "batch_files": 16,

    # local CV
    "oof_n_splits": 5   if MODE == "train" else 3,

    # dry-run
    "dryrun_n_files": 20 if MODE == "train" else 0,

    # train-only flags
    "run_oof": MODE == "train",
    "verbose": MODE == "train",

    # V18 proto_ssm
    "proto_ssm_train": {
        "n_epochs":        80  if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        20  if MODE == "train" else 8,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5   if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
    },
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,
        "n_epochs": 40  if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12  if MODE == "train" else 6,
    },
    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500  if MODE == "train" else 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20  if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },
}
print("✅ V18 CFG loaded")
print(f"  n_epochs={CFG['proto_ssm_train']['n_epochs']}  "
      f"patience={CFG['proto_ssm_train']['patience']}  "
      f"oof_n_splits={CFG['proto_ssm_train']['oof_n_splits']}  "
      f"mlp_max_iter={CFG['mlp_params']['max_iter']}")
 
print("Config ready")
print(f"  run_oof={CFG['run_oof']}  verbose={CFG['verbose']}  dryrun={CFG['dryrun_n_files']}")

import random as _random

def seed_everything(seed=42):
    import torch as _torch
    _random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark     = False

seed_everything(4)
print(f"Global random seed set to 4")


In [ ]:
# ── Cell 3: Data loading & label parsing ──────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
 
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
 
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
 
def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}
 
def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)
 
sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))
 
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
 
_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)
 
Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1
 
windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)
 
full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]
 
print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")


In [ ]:
# ── Cell 4: Load Perch model (ONNX preferred) ─────────────────────────
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

# ONNX session (150x faster)
ONNX_PERCH_PATH = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx")
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch (150x faster)")
else:
    print("Using TF SavedModel Perch")

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")


In [ ]:
# ── Cell 4b: Genus proxy logits for unmapped species ──────────────────
import re as _re

# Find which species have no direct Perch mapping
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

# For each unmapped species, find genus-level matches in Perch vocab
proxy_map = {}   # label_idx -> list of bc_indices

unmapped_df = (taxonomy[taxonomy["primary_label"]
               .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])]
               .copy())

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    
    # Find all Perch labels from the same genus
    hits = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(rf"^{_re.escape(genus)}\s", na=False)
    ]
    
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

# Only use proxies for biologically meaningful taxa
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped species total:        {len(UNMAPPED_POS)}")
print(f"Species with genus proxy:      {len(proxy_map)}")
print(f"Species still without signal:  {len(UNMAPPED_POS) - len(proxy_map)}")
print("\nProxy targets:")
for idx, bc_idxs in list(proxy_map.items())[:8]:
    label = PRIMARY_LABELS[idx]
    cls   = CLASS_NAME_MAP.get(label, "?")
    print(f"  {label:12s} ({cls:10s}) ← {len(bc_idxs)} Perch genus matches")


In [ ]:
# ── Cell 5: Perch inference engine (ONNX + multithreaded I/O) ─────────
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)

    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        # Prefetch first batch
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]

            # Prefetch next batch immediately
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS

            # ── ONNX or TF inference ───────────────────────────────────
            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("✅ Perch inference engine (ONNX + multithreaded I/O) defined")


In [ ]:
# ── Cell 6: Build-or-load Perch training cache ────────────────────────
print(f"USE_ONNX = {USE_ONNX}  "
      f"(cache will be built with {'ONNX' if USE_ONNX else 'TF SavedModel'})")

# Add any external cache locations here if you want to reuse pre-built data
EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]

CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"


def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None


SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]


def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k

    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k

    raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")


def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")

    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]

    t0 = time.time()

    meta_built, sc_built, emb_built = run_perch(
        train_paths,
        batch_files=CFG["batch_files"],
        verbose=True
    )

    print(f"  Perch pass done in {time.time()-t0:.1f}s  "
          f"scores={sc_built.shape} embs={emb_built.shape}")

    meta_built.to_parquet(CACHE_META_LOCAL)

    np.savez(
        CACHE_NPZ_LOCAL,
        scores=sc_built.astype(np.float32),
        embs=emb_built.astype(np.float32),
        primary_labels=np.array(PRIMARY_LABELS)
    )

    print(f"  Cache saved to {WORK_DIR}")

    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL


ext_meta, ext_npz = _find_external_cache()

if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")

elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")

else:
    print("No cache found — building from scratch (~1.5 min)")
    CACHE_META, CACHE_NPZ = _build_cache()


print("Loading Perch cache from:", CACHE_META.parent)

meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)


sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)

print(f"  scores ← '{sk}'  shape={sc_tr_raw.shape}")
print(f"  embs   ← '{ek}'  shape={emb_tr_raw.shape}")


sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)


if "primary_labels" in _arr.files:
    if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
        print("  WARNING: cached primary_labels differ — scores columns may not align!")
    else:
        print("  primary_labels schema OK")


if "row_id" not in meta_tr.columns:
    print("  row_id missing — reconstructing")

    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)

    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5

    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)

    meta_tr["row_id"] = (
        meta_tr["filename"].str.replace(".ogg", "", regex=False)
        + "_" + end_sec.astype(str)
    )


row_id_to_index = full_rows.set_index("row_id")["index"]

missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)

if missing_rows:
    raise RuntimeError(
        f"Cache has {len(missing_rows)} row_ids not in labeled set. "
        f"Delete {CACHE_META_LOCAL} and {CACHE_NPZ_LOCAL} to rebuild."
    )


Y_FULL_aligned = Y_SC[
    row_id_to_index.loc[meta_tr["row_id"]].to_numpy()
]

print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")


In [ ]:
# ── Cell 7: Metric helpers ─────────────────────────────────────────────
def macro_auc(y_true, y_score):
    """
    Exact replica of the competition metric:
    macro-averaged ROC-AUC, skipping classes with no positive labels.
    This is the ONLY number you should track locally.
    """
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")
 
 
def honest_oof_auc(scores, Y, meta_df, n_splits=5, label="scores"):
    """
    GroupKFold by filename — files never split across folds.
    This is the only correct way to estimate LB performance locally.
    Leaking a file across train/val inflates AUC by ~0.01–0.03.
    """
    groups = meta_df["filename"].to_numpy()
    gkf    = GroupKFold(n_splits=n_splits)
    oof    = np.zeros_like(scores, dtype=np.float32)
 
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(scores, groups=groups), 1):
        oof[va_idx] = scores[va_idx]
 
    auc = macro_auc(Y, oof)
    print(f"[{label}] honest OOF macro-AUC: {auc:.6f}")
    return auc, oof


In [ ]:
# ── Cell 7b: Temporal smoothing helper ─────────────────────────────────
def smooth_predictions(probs, n_windows=12, alpha=0.3):
    """
    For each file's 12 windows, blend each window with its neighbors.
    
    new[t] = (1 - alpha) * old[t] + 0.5*alpha * (old[t-1] + old[t+1])
    
    alpha=0: no smoothing (your current baseline)
    alpha=0.3: moderate smoothing (good starting point)
    
    Shape: (n_files * 12, n_classes) → same shape output
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"
    
    # Reshape to (n_files, 12, 234) so we can work file-by-file
    view = probs.reshape(-1, n_windows, C).copy()
    
    # Shift left and right (with edge padding = repeat boundary)
    prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)  # t-1
    next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)  # t+1
    
    smoothed = (1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)
    
    return smoothed.reshape(N, C)


print("✅ Temporal smoothing helper defined")


In [ ]:
# ── Cell 7c: Prior table builder ───────────────────────────────────────
def build_prior_tables(sc_df, Y_labels):
    """3-tier prior: global -> site/hour -> joint site x hour (shrinkage 4)."""
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)

    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)

    sh_keys = sorted({(str(s), int(h))
                      for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())})
    sh_to_i = {k: i for i, k in enumerate(sh_keys)}
    sh_p    = np.zeros((len(sh_keys), Y_labels.shape[1]), dtype=np.float32)
    sh_n    = np.zeros(len(sh_keys), dtype=np.float32)
    for sh_key in sh_keys:
        s, h = sh_key
        i = sh_to_i[sh_key]
        mask = (sc_df["site"].astype(str).values == s) & (sc_df["hour_utc"].astype(int).values == h)
        sh_n[i] = mask.sum(); sh_p[i] = Y_labels[mask].mean(axis=0)


    # Tweak D: circular Gaussian smoothing on hour priors (sigma=1.5, wrap-around)
    if len(hour_keys) >= 3:
        from scipy.ndimage import gaussian_filter1d as _gf1d
        _full = np.zeros((24, hour_p.shape[1]), dtype=np.float32)
        for _h, _i in hour_to_i.items():
            _full[int(_h)] = hour_p[_i]
        _tiled  = np.tile(_full, (3, 1))
        _smooth = _gf1d(_tiled, sigma=1.5, axis=0, mode='wrap')[24:48]
        for _h, _i in hour_to_i.items():
            hour_p[_i] = _smooth[int(_h)]
        hour_p = np.clip(hour_p, 0.0, 1.0)
    return {"global_p": global_p,
            "site_to_i": site_to_i, "site_p": site_p, "site_n": site_n,
            "hour_to_i": hour_to_i, "hour_p": hour_p, "hour_n": hour_n,
            "sh_to_i":   sh_to_i,   "sh_p":   sh_p,   "sh_n":   sh_n}


def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    eps = 1e-4
    out = scores.copy()
    p   = np.tile(tables["global_p"], (len(scores), 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]; nh = tables["hour_n"][j]
            p[i] = (nh/(nh+8.0))*tables["hour_p"][j] + (1-nh/(nh+8.0))*tables["global_p"]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]; ns = tables["site_n"][j]
            p[i] = (ns/(ns+8.0))*tables["site_p"][j] + (1-ns/(ns+8.0))*p[i]

    for i, (s, h) in enumerate(zip(sites, hours)):
        key = (str(s), int(h))
        if key in tables["sh_to_i"]:
            j = tables["sh_to_i"][key]; nsh = tables["sh_n"][j]
            p[i] = (nsh/(nsh+4.0))*tables["sh_p"][j] + (1-nsh/(nsh+4.0))*p[i]

    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)


print("Prior table functions defined — 3-tier joint site x hour (shrinkage 4)")


In [ ]:
# ── Cell 7d: File-level confidence scaling ─────────────────────────────
def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    """
    Scale each window's predictions by how confident the file is overall.
    
    Steps:
    1. For each file, find the top-k highest scores across all 12 windows
    2. Compute their mean → "file confidence"
    3. Multiply every window's scores by (file_confidence ** power)
    
    power=0: no effect (baseline)
    power=0.4: moderate suppression of uncertain files
    
    Why top-k and not max?
    Max is noisy (one lucky spike). Top-2 mean is more robust.
    """
    N, C = probs.shape
    assert N % n_windows == 0
    
    view      = probs.reshape(-1, n_windows, C)       # (n_files, 12, 234)
    sorted_v  = np.sort(view, axis=1)                 # sort across time
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)  # (n_files, 1, 234)
    
    scale  = np.power(top_k_mean, power)              # (n_files, 1, 234)
    scaled = view * scale                             # broadcast across 12 windows
    
    return scaled.reshape(N, C)


print("✅ File-level confidence scaling defined")


In [ ]:
# ── Cell 7e: Per-taxon temperature scaling ─────────────────────────────
# Build lookup: which species class are they?
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}   # continuous callers

# Build per-class temperature vector
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    if cls in TEXTURE_TAXA:
        temperatures[ci] = 0.95   # frogs/insects: slightly sharper
    else:
        temperatures[ci] = 1.10   # birds: slightly softer

n_texture = (temperatures < 1.0).sum()
n_event   = (temperatures > 1.0).sum()
print(f"✅ Temperatures: {n_event} event species (T=1.10), {n_texture} texture species (T=0.95)")


In [ ]:
# ── Cell 7f: UPGRADED MLP probe on PCA embeddings ─────────────────────
# CHANGE 1: Larger hidden layers (128,64), PCA 64-dim, max_iter=300
# Expected gain: +0.003–0.006 vs baseline (32,) hidden layer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def build_class_freq_weights(Y, cap=10.0):
    total     = Y.shape[0]
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / total
    weights   = 1.0 / (freq ** 0.5)
    weights   = np.clip(weights, 1.0, cap)
    weights   = weights / weights.mean()
    return weights.astype(np.float32)


def build_sequential_features(scores_col, n_windows=12):
    N = len(scores_col)
    assert N % n_windows == 0
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std


def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    """
    CHANGE 1: Upgraded MLP probe.
    - pca_dim: 32 → 64  (more embedding information)
    - hidden:  (32,) → (128, 64)  (more capacity)
    - max_iter: 100 → 300  (longer training)
    - min_pos: 8 → 5  (catches more rare species)
    """
    # Step 1: Compress embeddings
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding: {emb.shape} → PCA: {Z.shape}  "
          f"(variance retained: {pca.explained_variance_ratio_.sum():.2%})")

    class_weights = build_class_freq_weights(Y, cap=10.0)

    probe_models = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    print(f"Training MLP probes for {len(active)} species (>= {min_pos} pos windows)...")

    MAX_ROWS = 3000   # slightly higher budget for (128,64) layers

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue

        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([
            Z,
            scores_raw[:, ci:ci+1],
            prev[:, None], next_[:, None],
            mean[:, None], max_[:, None], std[:, None],
        ])

        n_pos = int(y.sum()); n_neg = len(y) - n_pos
        pos_idx = np.where(y == 1)[0]

        w      = float(class_weights[ci])
        repeat = max(1, int(round(w * n_neg / max(n_pos, 1))))
        repeat = min(repeat, 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))

        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])

        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64),   # CHANGE 1: was (32,)
            activation="relu",
            max_iter=300,                   # CHANGE 1: was 100
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,            # CHANGE 1: was 10
            random_state=42,
            learning_rate_init=5e-4,        # CHANGE 1: was 1e-3 (lower lr for deeper net)
            alpha=0.005,                    # CHANGE 1: was 0.01
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    print(f"Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend


def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4):
    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std = build_sequential_features(scores_test[:, ci])
        X_test = np.hstack([
            Z_test, scores_test[:, ci:ci+1],
            prev[:, None], next_[:, None],
            mean[:, None], max_[:, None], std[:, None],
        ])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print("✅ CHANGE 1: Upgraded MLP probe (pca_dim=64, hidden=(128,64), max_iter=300, min_pos=5)")


In [ ]:
# ── Cell 7f-2: Vectorized MLP probe inference ──────────────────────────
import torch
import torch.nn as nn

class VectorizedMLPProbes(nn.Module):
    """Stacks all per-class MLP weights into a single batched PyTorch model.
    Replaces the slow Python for-loop over probe_models at inference time."""
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.weights = nn.ParameterList()
            self.biases  = nn.ParameterList()
            self.n_layers = 0
            return

        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights  = nn.ParameterList()
        self.biases   = nn.ParameterList()

        for layer_idx in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[layer_idx]
                          for c in self.valid_classes], axis=0)       # (V, in, out)
            b = np.stack([probe_models[c].intercepts_[layer_idx]
                          for c in self.valid_classes], axis=0)       # (V, out)
            self.weights.append(nn.Parameter(
                torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(
                torch.tensor(b, dtype=torch.float32), requires_grad=False))

    def forward(self, x):
        # x: (V, N, in_dim)
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)   # (V, N)


def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models,
                                 scaler, pca, alpha_blend=0.4):
    """
    Drop-in replacement for apply_mlp_probes().
    Uses batched PyTorch matrix multiply instead of a Python for-loop —
    ~10-50x faster at inference time.
    """
    if len(probe_models) == 0:
        return scores_test.copy()

    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)

    valid_classes = sorted(probe_models.keys())
    V = len(valid_classes)
    N = len(scores_test)

    # Build sequential features for all classes at once
    raw  = scores_test[:, valid_classes].T          # (V, N)
    n_files = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx   = np.repeat(raw_view.max(axis=2),  N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)

    # scalar_feats: (V, N, 6)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)

    # Z_test: (N, D) → broadcast to (V, N, D)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))

    # X_all: (V, N, D+6)
    X_all = np.concatenate(
        [Z_expanded.astype(np.float32), scalar_feats], axis=-1)

    vec_probe = VectorizedMLPProbes(probe_models)
    vec_probe.eval()
    with torch.no_grad():
        preds = vec_probe(torch.tensor(X_all)).numpy()   # (V, N)

    result = scores_test.copy()
    base_valid = scores_test[:, valid_classes]           # (N, V)
    result[:, valid_classes] = (
        (1.0 - alpha_blend) * base_valid +
        alpha_blend * preds.T
    )
    return result

print("✅ Vectorized MLP probe inference defined")


In [ ]:
# ── Cell 7f-3: Isotonic Calibration + Per-Class Threshold Optimization ──
# CHANGE 2: Used by top notebooks (a.txt/d.txt), expected +0.004–0.008
# Trains isotonic regression per class on OOF scores to calibrate probs,
# then finds the best F1-threshold per species via grid search.
from sklearn.isotonic import IsotonicRegression

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, 
                                       threshold_grid=None, n_windows=12):
    """
    CHANGE 2: For each species:
    1. Fit isotonic regression on OOF scores (calibrates overconfident/underconfident classes)
    2. Grid-search F1-optimal threshold over calibrated probs
    Returns: thresholds array of shape (n_classes,)
    """
    if threshold_grid is None:
        threshold_grid = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    
    n_samples, n_cls = oof_probs.shape
    thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    n_files    = n_samples // n_windows
    file_oof   = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y     = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
    
    n_calibrated = 0
    for c in range(n_cls):
        y_true = file_y[:, c]
        y_prob = file_oof[:, c]
        if y_true.sum() < 3:
            continue
        try:
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except Exception:
            y_cal = y_prob
        
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp = ((pred==1) & (y_true==1)).sum()
            fp = ((pred==1) & (y_true==0)).sum()
            fn = ((pred==0) & (y_true==1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec  = tp / (tp + fn + 1e-8)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
        n_calibrated += 1
    
    print(f"Calibrated {n_calibrated} classes")
    print(f"Mean threshold: {thresholds.mean():.3f}")
    print(f"Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds


def apply_per_class_thresholds(scores, thresholds):
    """
    Sharpens probabilities around the per-class threshold:
    - above threshold → push toward 1
    - below threshold → push toward 0
    """
    C = scores.shape[1]
    assert C == len(thresholds)
    scaled = np.copy(scores)
    for c in range(C):
        t = thresholds[c]
        above = scores[:, c] > t
        scaled[ above, c] = 0.5 + 0.5 * (scores[ above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)

print("✅ CHANGE 2: Isotonic calibration + per-class threshold optimization defined")


In [ ]:
# ── Cell 7g: Rank-aware scaling ────────────────────────────────────────
def rank_aware_scaling(probs, n_windows=12, power=0.4):
    """
    CHANGE 6: Scale each window by the file's single peak confidence.

    How it works:
      1. For each file, find the MAX score across all 12 windows (per species)
      2. Raise it to power → scale factor
      3. Multiply every window's score by that scale factor

    Example for one species across 12 windows:
      Confident file:  max=0.90 → scale=0.90^0.4=0.96 → mild boost
      Uncertain file:  max=0.10 → scale=0.10^0.4=0.40 → strong suppression

    How this differs from Change 3 (file_confidence_scale):
      Change 3 uses top-2 MEAN → smoother, less aggressive
      Change 6 uses single MAX  → asks "does ANY window have strong evidence?"

    power=0.0 → no effect (baseline)
    power=0.4 → moderate suppression of uncertain files (recommended start)
    power=1.0 → multiply directly by file max (very aggressive)
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    view     = probs.reshape(-1, n_windows, C)              # (n_files, 12, 234)
    file_max = view.max(axis=1, keepdims=True)              # (n_files, 1, 234)

    scale  = np.power(file_max, power)                      # (n_files, 1, 234)
    scaled = view * scale                                   # broadcast to all 12 windows

    return scaled.reshape(N, C)


print("✅ Rank-aware scaling defined")


In [ ]:
# ── Cell 7h: Adaptive delta smoothing ─────────────────────────────────
def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    """
    CHANGE 7: Smooth uncertain windows toward their neighbors,
    while leaving confident windows almost untouched.

    How it works:
      For each window t:
        conf  = max probability across all 234 species at window t
        alpha = base_alpha * (1 - conf)   ← KEY: adapts to confidence
        new[t] = (1 - alpha) * old[t] + alpha * avg(old[t-1], old[t+1])

    Why alpha adapts to confidence:
      Confident window (max=0.90):
        alpha = 0.20 * (1 - 0.90) = 0.02  → barely smoothed, peak preserved
      Uncertain window (max=0.10):
        alpha = 0.20 * (1 - 0.10) = 0.18  → smoothed more, noise reduced

    This is exactly why your Change 1 hurt (-0.005) but this one should help:
      Change 1 used fixed alpha=0.3 → diluted confident peaks equally
      Change 7 uses adaptive alpha  → protects confident peaks, smooths noise

    base_alpha=0.0  → no smoothing (baseline)
    base_alpha=0.20 → recommended starting point
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)    # (n_files, 12, 234) original
    out    = result.reshape(-1, n_windows, C)   # (n_files, 12, 234) to modify

    for t in range(n_windows):

        # Confidence at this window = max prob across all species
        # Shape: (n_files, 1) — one confidence value per file per window
        conf = view[:, t, :].max(axis=-1, keepdims=True)   # (n_files, 1)

        # Adaptive alpha — low confidence = more smoothing
        alpha = base_alpha * (1.0 - conf)                  # (n_files, 1)

        # Neighbor average with edge padding
        if t == 0:
            # First window: left neighbor = itself
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            # Last window: right neighbor = itself
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0

        # Blend: confident windows barely change, uncertain ones smooth more
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg

    return result


print("✅ Adaptive delta smoothing defined")


In [ ]:
# ── Cell 7i: LightProtoSSM WITH Cross-Attention ────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model
        )
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(B_sz, D, self.d_state)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc  = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd  = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd  = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge= nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop     = nn.Dropout(dropout)

        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                      dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])

        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]

        for i, (fwd, bwd, merge, norm) in enumerate(zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)

        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None, None, :])
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            out   = alpha * sim + (1 - alpha) * perch_logits
        else:
            out = sim
        return out

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                           n_epochs=40, patience=8, lr=1e-3,
                           n_sites=20, verbose=False):
    """Train LightProtoSSM with cross-attention + SWA."""
    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fnames  = meta_full["filename"].unique()
    sites_u = sorted(meta_full["site"].unique())
    site2i  = {s: i + 1 for i, s in enumerate(sites_u)}

    site_ids = np.array([
        min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0], 0), n_sites-1)
        for fn in fnames], dtype=np.int64)
    hour_ids = np.array([
        int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0]) % 24
        for fn in fnames], dtype=np.int64)

    model = LightProtoSSM(n_classes=N_CLASSES, n_sites=n_sites,
                          use_cross_attn=True, cross_attn_heads=2)
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full,   dtype=torch.float32))
    print(f"LightProtoSSM params: {model.count_parameters():,}")

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    log_t  = torch.tensor(log_f,    dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,    dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0, 1))
    total      = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0

    # ── SWA setup ──────────────────────────────────────────────────────
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(n_epochs * 0.55)
    swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=3e-4)

    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (F.binary_cross_entropy_with_logits(
                    out, lab_t, pos_weight=pos_weight[None, None, :])
                + 0.15 * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        # ── SWA update ─────────────────────────────────────────────────
        if ep >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            sched.step()

        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    # ── Use SWA model if we reached swa_start, else best checkpoint ────
    if ep >= swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0), swa_model)
        model = swa_model
    else:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        out = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
    print(f"LightProtoSSM trained — best loss={best_loss:.4f}")
    return model, site2i


print("✅ CHANGE 4: LightProtoSSM with cross-attention (2 heads) + SWA defined")


In [ ]:
# ── Cell 7i-2: TTA — Circular Shift Test-Time Augmentation ───────────
# CHANGE 3: Average ProtoSSM predictions across 5 time shifts
# Expected gain: +0.003–0.005 on public LB

def run_tta_proto(proto_model, emb_files, sc_files,
                  site_t, hour_t, shifts=[0, 1, -1, 2, -2]):
    """
    CHANGE 3: TTA by circular-shifting 12-window sequences.
    
    For each shift s:
      1. Roll embeddings and perch logits by s windows
      2. Run ProtoSSM → get predictions
      3. Roll predictions back by -s (undo shift)
    
    Finally average all predictions across shifts.
    
    Why this works:
      - ProtoSSM sees temporal context across all 12 windows
      - Different starting points expose different context patterns
      - Averaging over 5 views reduces temporal boundary artifacts
    """
    proto_model.eval()
    all_preds = []
    
    emb_t  = torch.tensor(emb_files, dtype=torch.float32)
    sc_t   = torch.tensor(sc_files,  dtype=torch.float32)
    
    for shift in shifts:
        if shift == 0:
            e_shifted = emb_t
            s_shifted = sc_t
        else:
            e_shifted = torch.roll(emb_t, shift, dims=1)
            s_shifted = torch.roll(sc_t,  shift, dims=1)
        
        with torch.no_grad():
            out = proto_model(
                e_shifted, s_shifted,
                site_ids=site_t, hours=hour_t
            ).numpy()   # (n_files, 12, 234)
        
        if shift != 0:
            out = np.roll(out, -shift, axis=1)  # undo shift
        
        all_preds.append(out)
    
    # Tweak F: temporal flip as extra TTA pass
    with torch.no_grad():
        out_flip = proto_model(
            emb_t.flip(1), sc_t.flip(1),
            site_ids=site_t, hours=hour_t
        ).numpy()
    all_preds.append(out_flip[:, ::-1, :].copy())  # flip output back

    # Weighted TTA: shift=0 is ground truth, augmentations share 60%
    # Order: shift 0, +1, -1, +2, -2, flip (6 total)
    tta_w = np.array([0.40, 0.12, 0.12, 0.10, 0.10, 0.16], dtype=np.float32)
    return np.average(all_preds, axis=0, weights=tta_w)

print("✅ CHANGE 3: TTA with 5 circular shifts defined")


In [ ]:
# ── Cell 7j: Residual SSM (second-pass error correction) ──────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualSSM(nn.Module):
    """
    Lightweight second-pass model that learns to correct
    systematic errors from the first-pass ensemble.
    
    Input:  embeddings + first-pass scores (concatenated)
    Output: additive correction to first-pass scores
    
    Key design: output head initialized to zero
    so corrections start small and only grow if helpful.
    ~25s training on 59 files.
    """
    def __init__(self, d_input=1536, d_scores=234,
                 d_model=64, d_state=8,
                 n_classes=234, n_windows=12,
                 dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))

        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24,      meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(
            torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)
        # Zero init — corrections start at zero, only grow if helpful
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                 self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta.unsqueeze(1)

        res = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h   = self.ssm_drop(self.ssm_merge(
            torch.cat([h_f, h_b], dim=-1)))
        h   = self.ssm_norm(h + res)

        return self.output_head(h)   # (B, T, n_classes)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters()
                   if p.requires_grad)


def train_residual_ssm(emb_full, first_pass_flat, Y_full,
                       site_ids, hour_ids,
                       n_epochs=30, patience=8, lr=1e-3,
                       correction_weight=0.30,
                       verbose=False):
    """
    Train ResidualSSM to predict (Y - sigmoid(first_pass)).
    Returns corrected flat scores (n_rows, n_classes).
    ~20s on CPU.
    """
    n_files    = len(emb_full) // N_WINDOWS
    emb_f      = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f       = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f      = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    # Residual target = label - sigmoid(first_pass)
    fp_prob    = 1.0 / (1.0 + np.exp(-np.clip(fp_f, -30, 30)))
    residuals  = lab_f - fp_prob   # values in [-1, 1]

    print(f"Residuals: mean={residuals.mean():.4f}  "
          f"std={residuals.std():.4f}  "
          f"abs_mean={np.abs(residuals).mean():.4f}")

    # Train / val split (file level, no shuffle leakage)
    n_val    = max(1, int(n_files * 0.15))
    rng      = torch.Generator(); rng.manual_seed(42)
    perm     = torch.randperm(n_files, generator=rng).numpy()
    val_i    = perm[:n_val];  train_i = perm[n_val:]

    emb_t    = torch.tensor(emb_f,    dtype=torch.float32)
    fp_t     = torch.tensor(fp_f,     dtype=torch.float32)
    res_t    = torch.tensor(residuals, dtype=torch.float32)
    site_t   = torch.tensor(site_ids, dtype=torch.long)
    hour_t   = torch.tensor(hour_ids, dtype=torch.long)

    model    = ResidualSSM(n_classes=N_CLASSES)
    print(f"ResidualSSM params: {model.count_parameters():,}")

    opt      = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-3)
    sched    = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0

    for ep in range(n_epochs):
        model.train()
        corr = model(emb_t[train_i], fp_t[train_i],
                     site_ids=site_t[train_i],
                     hours   =hour_t[train_i])
        loss = F.mse_loss(corr, res_t[train_i])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        model.eval()
        with torch.no_grad():
            val_corr = model(emb_t[val_i], fp_t[val_i],
                             site_ids=site_t[val_i],
                             hours   =hour_t[val_i])
            val_loss = F.mse_loss(val_corr, res_t[val_i])

        if val_loss.item() < best_loss:
            best_loss  = val_loss.item()
            best_state = {k: v.clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    model.load_state_dict(best_state)
    print(f"ResidualSSM trained — best val MSE={best_loss:.6f}")

    # Apply correction to ALL training data (for verification)
    model.eval()
    with torch.no_grad():
        all_corr = model(emb_t, fp_t,
                         site_ids=site_t,
                         hours   =hour_t).numpy()
    print(f"Correction magnitude: "
          f"mean_abs={np.abs(all_corr).mean():.4f}  "
          f"max={np.abs(all_corr).max():.4f}")

    return model, correction_weight


print("✅ ResidualSSM defined (~439K params, ~20s training)")


In [ ]:
# ── Cell 8: OOF evaluation (train mode only) ──────────────────────────
baseline_auc = None
oof_raw      = None
 
if CFG["run_oof"]:
    print("Running honest OOF evaluation on training data…")
    baseline_auc, oof_raw = honest_oof_auc(
        sc_tr, Y_FULL_aligned, meta_tr,
        n_splits=CFG["oof_n_splits"],
        label="raw Perch"
    )
    print(f"\nBaseline OOF AUC: {baseline_auc:.6f}  ← your starting point")
else:
    print("Submit mode: skipping OOF evaluation")


In [ ]:
# ── Cell 8b: Full Pipeline OOF ─────────────────────────────────────────

def run_pipeline_oof(emb_full, sc_full, Y_full, meta_full, n_splits=5):
    """
    Proper full-pipeline OOF.
    Trains ProtoSSM + MLP on K-1 folds, predicts on held-out fold.
    ~3-4 min total on CPU. Use this instead of the raw-Perch OOF.
    """
    file_meta = (
        meta_full.drop_duplicates("filename")
        .reset_index(drop=True)
    )

    gkf = GroupKFold(n_splits=n_splits)
    oof_probs = np.zeros((len(sc_full), N_CLASSES), dtype=np.float32)

    for fold, (tr_f, va_f) in enumerate(
        gkf.split(file_meta, groups=file_meta["filename"]), 1
    ):
        tr_fnames = set(file_meta.iloc[tr_f]["filename"])
        va_fnames = set(file_meta.iloc[va_f]["filename"])

        tr_mask = meta_full["filename"].isin(tr_fnames).values
        va_mask = meta_full["filename"].isin(va_fnames).values

        emb_tr_f = emb_full[tr_mask]
        sc_tr_f = sc_full[tr_mask]
        Y_tr_f = Y_full[tr_mask]
        meta_tr_f = meta_full[tr_mask].reset_index(drop=True)

        emb_va_f = emb_full[va_mask]
        sc_va_f = sc_full[va_mask]
        meta_va_f = meta_full[va_mask].reset_index(drop=True)

        # ── Train ProtoSSM on train fold ───────────────────────────────
        proto_model, site2i = train_light_proto_ssm(
            emb_tr_f,
            sc_tr_f,
            Y_tr_f,
            meta_tr_f,
            n_epochs=40,
            patience=8,
            lr=1e-3,
            verbose=False,
        )

        # ── ProtoSSM predict on val fold ───────────────────────────────
        n_va = len(emb_va_f) // N_WINDOWS

        va_fn_list = (
            meta_va_f.drop_duplicates("filename")["filename"].tolist()
        )

        va_site_ids = np.array(
            [
                min(
                    site2i.get(
                        meta_va_f.loc[
                            meta_va_f["filename"] == fn, "site"
                        ].iloc[0],
                        0,
                    ),
                    19,
                )
                for fn in va_fn_list
            ],
            dtype=np.int64,
        )

        va_hour_ids = np.array(
            [
                int(
                    meta_va_f.loc[
                        meta_va_f["filename"] == fn, "hour_utc"
                    ].iloc[0]
                )
                % 24
                for fn in va_fn_list
            ],
            dtype=np.int64,
        )

        proto_model.eval()
        with torch.no_grad():
            proto_va = proto_model(
                torch.tensor(
                    emb_va_f.reshape(n_va, N_WINDOWS, -1),
                    dtype=torch.float32,
                ),
                torch.tensor(
                    sc_va_f.reshape(n_va, N_WINDOWS, -1),
                    dtype=torch.float32,
                ),
                site_ids=torch.tensor(va_site_ids, dtype=torch.long),
                hours=torch.tensor(va_hour_ids, dtype=torch.long),
            ).numpy().reshape(-1, N_CLASSES)

        # ── Train MLP on train fold ────────────────────────────────────
        probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
            emb_tr_f,
            sc_tr_f,
            Y_tr_f,
            min_pos=5,
            pca_dim=64,
            alpha_blend=0.4,
        )

        sc_va_mlp = apply_mlp_probes_vectorized(
            emb_va_f,
            sc_va_f,
            probe_models,
            emb_scaler,
            emb_pca,
            alpha_blend,
        )

        # ── Ensemble + sigmoid ─────────────────────────────────────────
        first_pass = 0.5 * proto_va + 0.5 * sc_va_mlp
        probs_va = 1.0 / (1.0 + np.exp(-np.clip(first_pass, -30, 30)))
        oof_probs[va_mask] = probs_va

        fold_auc = macro_auc(Y_full[va_mask], probs_va)
        print(
            f"  Fold {fold}/{n_splits}  val files={len(va_fnames)}  AUC={fold_auc:.6f}"
        )

    overall = macro_auc(Y_full, oof_probs)
    print(f"\nFull pipeline OOF AUC: {overall:.6f}")
    return overall, oof_probs


if CFG["run_oof"]:
    pipeline_auc, oof_pipeline = run_pipeline_oof(
        emb_tr,
        sc_tr,
        Y_FULL_aligned,
        meta_tr,
        n_splits=5,
    )


In [ ]:
# ── Cell 9: Test inference ─────────────────────────────────────────────
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
 
if not test_paths:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")
 
meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"Test scores: {sc_te.shape}")


In [ ]:
# ── Cell 10: Full pipeline with ProtoSSM + ResidualSSM ─────────────────

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# ── Step A: Train LightProtoSSM ────────────────────────────────────────
t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=70, patience=12, lr=1e-3, verbose=False)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

# ── Step B: Run ProtoSSM on TEST ───────────────────────────────────────
n_test_files  = len(sc_te) // N_WINDOWS
emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
n_sites_cap   = 20
test_site_ids = np.array([
    min(site2i_tr.get(
        meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0], 0),
        n_sites_cap-1)
    for fn in test_fnames], dtype=np.int64)
test_hour_ids = np.array([
    int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0]) % 24
    for fn in test_fnames], dtype=np.int64)

proto_out = run_tta_proto(
    proto_model, emb_te_f, sc_te_f,
    site_t=torch.tensor(test_site_ids, dtype=torch.long),
    hour_t=torch.tensor(test_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2],
)
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

# ── Step C: Prior tables ───────────────────────────────────────────────
prior_tables   = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(
    sc_te,
    sites=meta_te["site"].to_numpy(),
    hours=meta_te["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)

# ── Step D: MLP probes ─────────────────────────────────────────────────
probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned,
    min_pos=5, pca_dim=64, alpha_blend=0.4,
)
sc_te_adjusted = apply_mlp_probes_vectorized(
    emb_te, sc_te_adjusted,
    probe_models, emb_scaler, emb_pca, alpha_blend,
)

# ── Step E: First-pass ensemble (ProtoSSM + MLP) ───────────────────────
# Tweak A: per-class weights — MAPPED species trust ProtoSSM more
ENSEMBLE_W_PER_CLASS = np.where(MAPPED_MASK, 0.60, 0.35).astype(np.float32)
first_pass_flat = (ENSEMBLE_W_PER_CLASS[None, :] * proto_scores_flat
                   + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_te_adjusted)

# ── Step F: ResidualSSM (second-pass correction) ───────────────────────
# Build training-data first-pass scores for residual training
n_tr_files    = len(sc_tr) // N_WINDOWS
emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
tr_site_ids   = np.array([
    min(site2i_tr.get(
        meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0], 0),
        n_sites_cap-1)
    for fn in tr_fnames], dtype=np.int64)
tr_hour_ids   = np.array([
    int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0]) % 24
    for fn in tr_fnames], dtype=np.int64)


# Get ProtoSSM scores on training data
# CORRECT — using emb_tr_f, sc_tr_f, tr_site_ids (train data)
proto_tr_out = run_tta_proto(
    proto_model, emb_tr_f, sc_tr_f,
    site_t=torch.tensor(tr_site_ids, dtype=torch.long),
    hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2],
)

proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

# Get MLP scores on training data
sc_tr_prior   = apply_prior(
    sc_tr,
    sites=meta_tr["site"].to_numpy(),
    hours=meta_tr["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)
sc_tr_mlp = apply_mlp_probes_vectorized(
    emb_tr, sc_tr_prior,
    probe_models, emb_scaler, emb_pca, alpha_blend,
)
first_pass_tr = (ENSEMBLE_W_PER_CLASS[None, :] * proto_tr_flat
                 + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_tr_mlp)

train_probs_for_calib = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_for_calib,
    Y_FULL=Y_FULL_aligned,
    threshold_grid=([round(t,3) for t in np.arange(0.20, 0.45, 0.025)]
                         + [round(t,3) for t in np.arange(0.45, 0.75, 0.05)]),
    n_windows=N_WINDOWS,
)


# Train ResidualSSM on training errors
t0 = time.time()
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr,
    first_pass_flat=first_pass_tr,
    Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids,
    hour_ids=tr_hour_ids,
    n_epochs=30,
    patience=8,
    lr=1e-3,
    correction_weight=0.30,
    verbose=False,
)
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

# Tweak C: grid search for best correction_weight on training residuals
_CORRECTION_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
_emb_tr_f_c = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
_fp_tr_f_c  = first_pass_tr.reshape(n_tr_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    _tr_correction = res_model(
        torch.tensor(_emb_tr_f_c, dtype=torch.float32),
        torch.tensor(_fp_tr_f_c,  dtype=torch.float32),
        site_ids=torch.tensor(tr_site_ids, dtype=torch.long),
        hours   =torch.tensor(tr_hour_ids, dtype=torch.long),
    ).numpy().reshape(-1, N_CLASSES).astype(np.float32)
_best_auc, _best_w = -1.0, 0.30
for _w in _CORRECTION_GRID:
    _trial_probs = sigmoid(first_pass_tr + _w * _tr_correction)
    _auc = macro_auc(Y_FULL_aligned, _trial_probs)
    print(f"  correction_weight={_w:.2f}  train AUC={_auc:.5f}")
    if _auc > _best_auc:
        _best_auc, _best_w = _auc, _w
correction_weight = _best_w
print(f"Best correction_weight={correction_weight:.2f}  (AUC={_best_auc:.5f})")
del _emb_tr_f_c, _fp_tr_f_c, _tr_correction

# Apply ResidualSSM correction to TEST scores
first_pass_te_f  = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f,         dtype=torch.float32),
        torch.tensor(first_pass_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()

correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = (first_pass_flat
                   + correction_weight * correction_flat)

print(f"Correction applied — "
      f"mean_abs={np.abs(correction_flat).mean():.4f}  "
      f"score range [{final_scores.min():.3f}, {final_scores.max():.3f}]")

# ── Step G: Temperature scaling ────────────────────────────────────────
final_scores = final_scores / temperatures[None, :]

# ── Step H: Sigmoid → probabilities ───────────────────────────────────
probs = sigmoid(final_scores)

# ── Step I: Post-processing pipeline ──────────────────────────────────
probs = file_confidence_scale(probs, n_windows=N_WINDOWS,
                               top_k=2,       power=0.4)
probs = rank_aware_scaling(   probs, n_windows=N_WINDOWS,
                               power=0.4)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS,
                               base_alpha=0.20)
probs = np.clip(probs, 0.0, 1.0)

probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)

# ── Step J: Build submission ───────────────────────────────────────────
sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub.insert(0, "row_id", meta_te["row_id"].values)
assert list(sub.columns) == ["row_id"] + PRIMARY_LABELS
assert len(sub) == len(test_paths) * N_WINDOWS
assert not sub.isna().any().any()
sub.to_csv("submission_protossm.csv", index=False)                                                                                        
protossm_sub = sub.copy()

print(f"\nsubmission.csv saved — shape {sub.shape}")
print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")

del emb_tr_f, sc_tr_f, proto_model, res_model                                                                                             
gc.collect()                                                                                                                              
print("Memory freed. Ready for SED cell.")


In [ ]:
# ── Cell 11: Tucker Arrants distilled SED ONNX inference ──────────────

import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80


def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError(
            "sed_fold0.onnx not found. "
            "Attach tuckerarrants/bc2026-distilled-sed-public to this notebook."
        )
    return hits[0].parent


def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    return ort.InferenceSession(
        str(path),
        sess_options=so,
        providers=["CPUExecutionProvider"]
    )


def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0,
        )
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)

    return np.stack(mels)[:, None].astype(np.float32)


def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)

    if y.ndim == 2:
        y = y.mean(axis=1)

    if sr0 != SR:
        y = librosa.resample(y, orig_sr=sr0, target_sr=SR)

    n = 60 * SR

    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    else:
        y = y[:n]

    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC

    return chunks, ends


def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


# Load the 5 SED fold models
sed_dir = find_sed_dir()

sed_fold_paths = sorted(
    sed_dir.glob("sed_fold*.onnx"),
    key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1))
)

sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")


# Run on the exact same test files used by Cell 9/10
sed_rows, sed_preds = [], []
_t0_sed = time.time()


for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel = audio_to_mel(chunks)

    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)

    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})

        clip_logits = outs[0]             # (12, 234)
        frame_max   = outs[1].max(axis=1) # (12, 234)

        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

    p_mean = p_sum / len(sed_sessions)

    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(
            p_mean,
            sigma=0.65,
            axis=0,
            mode="nearest"
        ).astype(np.float32)

    stem = path.stem

    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)

    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)} | {time.time()-_t0_sed:.1f}s")


sed_preds_arr = np.concatenate(sed_preds, axis=0)

sed_sub = pd.DataFrame(
    np.clip(sed_preds_arr, 0.0, 1.0),
    columns=PRIMARY_LABELS
)

sed_sub.insert(0, "row_id", sed_rows)

sed_sub.to_csv("submission_sed.csv", index=False)

print(f"Saved submission_sed.csv: {sed_sub.shape}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SUBMISSION MODE TOGGLE  ← Change this to 1, 2, or 3
# ═══════════════════════════════════════════════════════════════════
SUBMISSION_MODE = 3   # 1 = Optimized Baseline | 2 = Vertical Split | 3 = Extreme Sharpening

assert SUBMISSION_MODE in (1, 2, 3), "SUBMISSION_MODE must be 1, 2, or 3"

# ── Shared constants (do not edit) ──────────────────────────────────
PROTOSSM_CSV = "submission_protossm.csv"   # output of ProtoSSM inference cell
SED_CSV      = "submission_sed.csv"        # output of distilled-SED inference cell
OUT_CSV      = "submission.csv"            # final output
N_WINDOWS    = 12                          # 12 × 5s windows per 60s file
EPS          = 1e-6

# ── Mode-specific hyperparameters ───────────────────────────────────
# MODE 1 — Optimized Baseline
# Rationale: ProtoSSM is architecturally richer (ResidualSSM + MLP probes +
# genus-proxy expansion), so nudging from 0.60→0.65 is a low-risk gain.
# Power α=1.2 provides mild sharpening that kills borderline false positives
# while preserving high-confidence detections. Validated by: the 0.948
# baseline uses 0.60/0.40 raw without power; a 5-point weight shift toward
# the stronger model + minimal sharpening is the safest lever to pull.
M1 = dict(
    w_proto      = 0.65,   # ProtoSSM rank weight (baseline 0.60)
    w_sed        = 0.35,   # SED rank weight       (baseline 0.40)
    power_alpha  = 1.20,   # Moderate post-blend power sharpening
    file_top_k   = 2,      # top-k windows for file-level confidence scaling
    file_power   = 0.40,   # rank-aware file amplification exponent
    noise_thr_p  = 0.50,   # p_proto threshold for fake-only gate
    noise_thr_s  = 0.05,   # p_sed  threshold for fake-only gate
    noise_pull   = 0.08,   # pull weight toward rank_proto on gate trigger
)

# MODE 2 — Risky Vertical Split (EoS-4 inspired)
# Rationale: EoS-4 showed that the two models complement each other
# differently depending on WHERE in the test set a recording falls.
# Early recordings (first half, sorted by filename = earlier timestamps)
# tend to capture dawn chorus; ProtoSSM (hour-prior-aware) is stronger.
# Later recordings capture midday/afternoon; SED (mel-spectrogram based)
# handles lower-SNR broad-band vocalizations better. We also shift the
# intra-recording window cutoff from 6→5 (i.e., first 25s vs last 35s)
# because dawn chorus peaks in the first ~25s of most recordings.
# The per-half power differs: 1.15 for first (sharper on dawn peaks),
# 1.10 for second (gentler to preserve faint midday calls).
M2 = dict(
    # Inter-file vertical split: how many FILES go into "first half"
    # (None = 50/50 auto-split at the median file)
    file_boundary_frac  = 0.50,   # first N*frac files → first-half weights
    # Intra-recording window cutoff (1-indexed; windows 1..cutoff = first half)
    window_cutoff       = 5,      # shifted left (vs 6) to favor dawn windows
    # First-half blend weights
    w_proto_first       = 0.70,   # heavier ProtoSSM — dawn chorus
    w_sed_first         = 0.30,
    power_alpha_first   = 1.15,
    # Second-half blend weights
    w_proto_second      = 0.55,   # more SED — midday ambient
    w_sed_second        = 0.45,
    power_alpha_second  = 1.10,
    # Noise gate (shared)
    noise_thr_p         = 0.50,
    noise_thr_s         = 0.05,
    noise_pull          = 0.08,
)

# MODE 3 — Extreme Sharpening & Rank (High-Risk / High-Reward)
# Rationale: AUC is maximized by rank ordering, not calibration. If ProtoSSM
# is the dominant signal (75% weight) and we apply aggressive power α=1.5,
# low-confidence "background hum" detections get crushed toward 0 while
# high-confidence true positives get amplified. The risk is over-suppression
# of genuinely quiet calls. Offset by: (a) keeping SED spike preservation
# gate active at a lower threshold (0.90 vs 0.95), and (b) using a stronger
# file-level rank amplifier (top-k=3, file_power=0.50) to reward files where
# the model is consistently confident — a reliable proxy for true presence.
M3 = dict(
    w_proto      = 0.75,   # Heavy ProtoSSM bias
    w_sed        = 0.25,
    power_alpha  = 1.50,   # Aggressive sharpening
    file_top_k   = 3,      # top-3 windows for file confidence
    file_power   = 0.50,   # stronger rank-aware amplification
    noise_thr_p  = 0.55,   # tighter fake-only gate
    noise_thr_s  = 0.04,
    noise_pull   = 0.12,   # harder penalisation on trigger
    sed_spike_thr     = 0.90,   # lower threshold: catch more SED spikes
    proto_spike_floor = 0.75,   # proto must be genuinely low to let SED in
)

print(f"SUBMISSION_MODE = {SUBMISSION_MODE}")
if SUBMISSION_MODE == 1: print("Strategy: Optimized Baseline — 65/35 + Power 1.2")
if SUBMISSION_MODE == 2: print("Strategy: Risky Vertical Split — asymmetric half-weights + shifted window cutoff")
if SUBMISSION_MODE == 3: print("Strategy: Extreme Sharpening — 75/25 + Power 1.5 + file-level rank amplify")


## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")


## Cell 3 — Core Fusion Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# R  — Class-wise percentile rank transform
# Maps each class column independently to [0, 1] using fractional ranks.
# This removes calibration mismatch between ProtoSSM and SED: a SED score
# of 0.3 and a ProtoSSM score of 0.3 may not be equivalent in probability
# space, but rank=0.90 means the same thing (top 10%) for both models.
# ─────────────────────────────────────────────────────────────────────────
def classwise_percentile_rank(arr: np.ndarray) -> np.ndarray:
    """
    arr : (N_rows, N_classes)  float32
    returns: same shape, values in [1/(N+1), N/(N+1)]  (fractional ranks)
    Memory-efficient: uses scipy.stats.rankdata column-by-column in chunks.
    """
    N, C = arr.shape
    out = np.empty_like(arr)
    CHUNK = 64  # process 64 columns at a time to stay cache-friendly
    for start in range(0, C, CHUNK):
        end = min(start + CHUNK, C)
        chunk = arr[:, start:end]  # (N, chunk_size)
        # rankdata operates on 1-D arrays; apply over axis=0 via loop
        for ci in range(end - start):
            out[:, start + ci] = rankdata(chunk[:, ci], method="average") / (N + 1)
    return out


# ─────────────────────────────────────────────────────────────────────────
# G — Power sharpening (post-blend)
# Applies p^alpha element-wise.  alpha > 1 shrinks low probabilities faster
# than high ones, effectively widening the gap between true/false positives.
# Applied AFTER rank blending, before post-processing gates.
# ─────────────────────────────────────────────────────────────────────────
def power_sharpen(pred: np.ndarray, alpha: float) -> np.ndarray:
    """
    pred  : (N, C) float32, values in [0, 1]
    alpha : sharpening exponent (1.0 = no-op, >1 = sharpen)
    """
    return np.power(np.clip(pred, 0.0, 1.0), alpha).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────
# File-level rank-aware amplification
# Within each 60-second recording, compute a file-level "confidence signal"
# as the mean of the top-k window predictions.  Multiply each window's
# prediction by confidence^file_power.  This amplifies detections in files
# where the model is consistently confident — a proxy for true presence.
# ─────────────────────────────────────────────────────────────────────────
def file_rank_amplify(
    pred: np.ndarray,
    file_ids: np.ndarray,
    n_windows: int = 12,
    top_k: int = 2,
    file_power: float = 0.40,
) -> np.ndarray:
    """
    pred     : (N, C) float32
    file_ids : (N,)  array of file identifiers (repeated n_windows times each)
    """
    out = pred.copy()
    for fid in pd.unique(file_ids):
        m = file_ids == fid
        x = pred[m]                          # (n_windows, C)
        sorted_x = np.sort(x, axis=0)        # ascending along window axis
        top_k_clamped = min(top_k, x.shape[0])
        # Mean of the top-k window scores per class
        top_k_mean = sorted_x[-top_k_clamped:, :].mean(axis=0, keepdims=True)  # (1, C)
        scale = np.power(np.clip(top_k_mean, 0.0, 1.0), file_power)            # (1, C)
        out[m] = x * scale
    return out.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────
# Vertical split blend (EoS-4 inspired)
# Splits the test dataframe into two halves by file order and applies
# different rank-blend weights to each half.  Also supports an intra-file
# window cutoff so that early windows (dawn) vs late windows (midday)
# within the same recording can receive different treatment.
# ─────────────────────────────────────────────────────────────────────────
def vertical_split_blend(
    rank_proto: np.ndarray,
    rank_sed: np.ndarray,
    row_ids: np.ndarray,
    file_ids: np.ndarray,
    file_boundary_frac: float = 0.50,
    window_cutoff: int = 6,
    w_proto_first: float  = 0.70,
    w_sed_first: float    = 0.30,
    w_proto_second: float = 0.55,
    w_sed_second: float   = 0.45,
    power_alpha_first: float  = 1.15,
    power_alpha_second: float = 1.10,
) -> np.ndarray:
    """
    Splits rows into (inter-file first half) × (intra-file first windows)
    and applies different blend weights to each quadrant.

    Quadrant matrix:
                       | file first-half | file second-half |
      window first-half|  w1 (strongest) |  w3 (moderate)  |
      window second-half|  w2 (moderate) |  w4 (weakest)   |

    For simplicity we use two blend configs:
      rows where file is in first_half_files → (w_proto_first, w_sed_first)
      rows where file is in second_half_files → (w_proto_second, w_sed_second)
    with the window_cutoff providing an additional intra-file asymmetry.
    """
    unique_files = pd.unique(file_ids)  # preserves order
    n_files = len(unique_files)
    boundary = max(1, int(np.round(n_files * file_boundary_frac)))
    first_half_files  = set(unique_files[:boundary])
    second_half_files = set(unique_files[boundary:])

    pred = np.zeros_like(rank_proto)

    for fid in unique_files:
        m = file_ids == fid
        idxs = np.where(m)[0]          # row indices belonging to this file
        n_w  = len(idxs)               # should be N_WINDOWS

        # Intra-file split: first  windows vs remainder
        w_cutoff_clamped = min(window_cutoff, n_w)
        early_idx = idxs[:w_cutoff_clamped]
        late_idx  = idxs[w_cutoff_clamped:]

        if fid in first_half_files:
            wp, ws, pa = w_proto_first, w_sed_first, power_alpha_first
        else:
            wp, ws, pa = w_proto_second, w_sed_second, power_alpha_second

        # Early windows: slightly more ProtoSSM weight (dawn bias)
        early_blend = wp * rank_proto[early_idx] + ws * rank_sed[early_idx]
        pred[early_idx] = power_sharpen(early_blend, pa)

        if len(late_idx) > 0:
            # Late windows: shift 3pp toward SED
            wp_late = max(0.0, wp - 0.03)
            ws_late = min(1.0, ws + 0.03)
            pa_late = max(1.0, pa - 0.05)
            late_blend = wp_late * rank_proto[late_idx] + ws_late * rank_sed[late_idx]
            pred[late_idx] = power_sharpen(late_blend, pa_late)

    return pred.astype(np.float32)


print("Core fusion functions defined.")


## Cell 4 — Post-Processing Gates

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Gate 1: Noise suppression
# If ProtoSSM is confident but SED strongly disagrees → pull pred toward
# rank_proto to suppress likely false positives.
# ─────────────────────────────────────────────────────────────────────────
def gate_noise_suppression(
    pred, rank_proto, p_proto, p_sed,
    thr_p=0.50, thr_s=0.05, pull=0.08
):
    fake_only = (p_proto > thr_p) & (p_sed < thr_s)
    pred = np.where(fake_only, (1.0 - pull) * pred + pull * rank_proto, pred)
    return pred, fake_only


# ─────────────────────────────────────────────────────────────────────────
# Gate 2: Temporal continuity (fat-tailed kernel over ±3 windows = 35s)
# Smooths ProtoSSM predictions with a t-distribution kernel.  Continuous
# callers (e.g. insects, frogs) produce a smooth context signal; isolated
# spikes are NOT amplified.  The gate then slightly boosts rows where the
# smoothed rank is high AND raw proto rank is high but SED missed the call.
# ─────────────────────────────────────────────────────────────────────────
def gate_temporal_continuity(
    pred, rank_proto, rank_sed, p_proto, p_sed,
    file_ids, fake_only,
    ctx_pull=0.15, rank_ctx_thr=0.88, rank_proto_thr=0.75, p_sed_thr=0.12
):
    # Build fat-tailed t-distribution kernel  (df=3, half-width=3 windows)
    offs = np.arange(-3, 4, dtype=np.float32)   # [-3,-2,-1,0,1,2,3]
    kernel = (1.0 + (offs / 1.20) ** 2 / 3.0) ** (-2.0)  # t-dist pdf shape
    kernel = (kernel / kernel.sum()).astype(np.float32)    # normalise

    # Apply kernel file-by-file
    pa_ctx = p_proto.copy()
    for fid in pd.unique(file_ids):
        m = file_ids == fid
        x = p_proto[m]                                    # (nw, C)
        nw = x.shape[0]
        if nw < 2:
            continue
        xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
        smoothed = sum(kernel[i] * xp[i: i + nw] for i in range(7))
        pa_ctx[m] = smoothed

    rank_ctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
    proto_cont = (
        (rank_ctx   > rank_ctx_thr)   &
        (rank_proto > rank_proto_thr) &
        (p_sed      < p_sed_thr)      &
        (~fake_only)
    )
    pred = np.where(
        proto_cont,
        (1.0 - ctx_pull) * pred + ctx_pull * np.maximum(rank_proto, rank_ctx),
        pred,
    )
    return pred, proto_cont


# ─────────────────────────────────────────────────────────────────────────
# Gate 3: SED spike preservation
# Brief high-confidence SED detections that ProtoSSM missed.
# SED operates on mel-spectrograms and can catch sharp transient events
# (e.g. brief frog calls) that the SSM smooth representation averages out.
# ─────────────────────────────────────────────────────────────────────────
def gate_sed_spike(
    pred, rank_proto, rank_sed,
    fake_only, proto_cont,
    sed_spike_thr=0.95, proto_floor=0.80, spike_pull=0.12
):
    sed_only = (
        (rank_sed   > sed_spike_thr) &
        (rank_proto < proto_floor)   &
        (~fake_only) & (~proto_cont)
    )
    pred = np.where(sed_only, (1.0 - spike_pull) * pred + spike_pull * rank_sed, pred)
    return pred, sed_only


# ─────────────────────────────────────────────────────────────────────────
# Gate 4: Sonotype mirroring
# Max-pool across visually/acoustically identical species groups.
# These are sonotypes that are indistinguishable even to experts;
# assigning the group maximum to all members is the safest strategy.
# ─────────────────────────────────────────────────────────────────────────
MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)

def gate_sonotype_mirror(sub_df, cols):
    col_to_idx = {l: i for i, l in enumerate(cols)}
    mirror_count = 0
    for group in MIRROR_PAIRS:
        valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
        if len(valid_idx) >= 2:
            group_max = sub_df[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
            for idx in valid_idx:
                sub_df.iloc[:, idx + 1] = group_max
            mirror_count += len(valid_idx)
    print(f"Sonotype mirroring: {mirror_count} columns synced.")
    return sub_df


# ─────────────────────────────────────────────────────────────────────────
# Gate 5: Adaptive rare-class suppression
# Amphibia / Mammalia / Reptilia produce background texture that bleeds
# into many windows.  A mild threshold pulls sub-mean predictions down
# without touching top detections.
# ─────────────────────────────────────────────────────────────────────────
def gate_rare_class_suppress(sub_df, cols, taxonomy_path, suppress_factor=0.90, margin=0.05):
    try:
        tax_df = pd.read_csv(taxonomy_path).set_index("primary_label")
        rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
        count = 0
        for ci, species in enumerate(cols):
            if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
                col_idx = ci + 1  # +1 for row_id column
                vals = sub_df.iloc[:, col_idx].to_numpy(np.float32)
                thr = vals.mean() + margin
                sub_df.iloc[:, col_idx] = np.where(vals < thr, vals * suppress_factor, vals)
                count += 1
        print(f"Rare-class suppression: {count} species adjusted.")
    except Exception as e:
        print(f"Rare-class suppression skipped: {e}")
    return sub_df


print("Post-processing gates defined.")


## Cell 5 — Load Intermediate Predictions

In [ ]:
df_proto = pd.read_csv(PROTOSSM_CSV)
df_sed   = pd.read_csv(SED_CSV)

print(f"ProtoSSM shape : {df_proto.shape}")
print(f"SED shape      : {df_sed.shape}")

# Align row order to ProtoSSM (authoritative row_id ordering)
df_sed = df_sed.set_index("row_id").loc[df_proto["row_id"]].reset_index()

cols     = [c for c in df_proto.columns if c != "row_id"]  # 234 class columns
row_ids  = df_proto["row_id"].astype(str).to_numpy()

# Derive file_ids: everything before the last underscore
# e.g. "BC2026_Test_001_S01_20240101_060000_5" → "BC2026_Test_001_S01_20240101_060000"
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

# Raw probability arrays (clipped to avoid log/power edge cases)
p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
p_sed   = np.clip(df_sed[cols].to_numpy(np.float32),   EPS, 1.0 - EPS)

# R — class-wise percentile rank transform
print("Computing class-wise percentile ranks...")
rank_proto = classwise_percentile_rank(p_proto)  # (N, 234)
rank_sed   = classwise_percentile_rank(p_sed)    # (N, 234)
print(f"rank_proto: min={rank_proto.min():.4f}  max={rank_proto.max():.4f}")
print(f"rank_sed  : min={rank_sed.min():.4f}  max={rank_sed.max():.4f}")


## Cell 6 — Mode-Specific Blend

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# MODE 1: Optimized Baseline
#   pred = power( 0.65·R(proto) + 0.35·R(sed), α=1.2 )
#   + file-level rank amplification (top-2, power=0.40)
# ─────────────────────────────────────────────────────────────────────────
if SUBMISSION_MODE == 1:
    p = M1
    print(f"Mode 1 | weights: proto={p["w_proto"]} sed={p["w_sed"]} | "
          f"power={p["power_alpha"]} | file_top_k={p["file_top_k"]} file_power={p["file_power"]}")

    # Step 1: weighted rank blend
    pred = p["w_proto"] * rank_proto + p["w_sed"] * rank_sed  # (N, C)

    # Step 2: power sharpening
    pred = power_sharpen(pred, p["power_alpha"])

    # Step 3: file-level rank amplification
    #   Rewards files where the model is consistently high-confidence
    pred = file_rank_amplify(
        pred, file_ids,
        n_windows=N_WINDOWS,
        top_k=p["file_top_k"],
        file_power=p["file_power"],
    )

    # Step 4: re-normalise to [0,1] after amplification
    pred = pred / (pred.max() + EPS)

    noise_thr_p = p["noise_thr_p"]
    noise_thr_s = p["noise_thr_s"]
    noise_pull  = p["noise_pull"]
    sed_spike_thr    = 0.95
    proto_spike_floor = 0.80


# ─────────────────────────────────────────────────────────────────────────
# MODE 2: Risky Vertical Split
#   First/last halves of test files get different blend weights.
#   Within each file, early windows (≤cutoff) vs late windows differ by 3pp.
# ─────────────────────────────────────────────────────────────────────────
elif SUBMISSION_MODE == 2:
    p = M2
    unique_files = pd.unique(file_ids)
    n_files = len(unique_files)
    boundary = max(1, int(np.round(n_files * p["file_boundary_frac"])))
    print(f"Mode 2 | {n_files} test files | boundary @ file #{boundary} "
          f"| window_cutoff={p["window_cutoff"]}")
    print(f"  first-half  weights: proto={p["w_proto_first"]} sed={p["w_sed_first"]} "
          f"power={p["power_alpha_first"]}")
    print(f"  second-half weights: proto={p["w_proto_second"]} sed={p["w_sed_second"]} "
          f"power={p["power_alpha_second"]}")

    pred = vertical_split_blend(
        rank_proto=rank_proto,
        rank_sed=rank_sed,
        row_ids=row_ids,
        file_ids=file_ids,
        file_boundary_frac=p["file_boundary_frac"],
        window_cutoff=p["window_cutoff"],
        w_proto_first=p["w_proto_first"],
        w_sed_first=p["w_sed_first"],
        w_proto_second=p["w_proto_second"],
        w_sed_second=p["w_sed_second"],
        power_alpha_first=p["power_alpha_first"],
        power_alpha_second=p["power_alpha_second"],
    )

    # File-level amplification with moderate settings
    pred = file_rank_amplify(pred, file_ids, n_windows=N_WINDOWS, top_k=2, file_power=0.35)
    pred = pred / (pred.max() + EPS)

    noise_thr_p = p["noise_thr_p"]
    noise_thr_s = p["noise_thr_s"]
    noise_pull  = p["noise_pull"]
    sed_spike_thr    = 0.95
    proto_spike_floor = 0.80


# ─────────────────────────────────────────────────────────────────────────
# MODE 3: Extreme Sharpening & Rank
#   pred = power( 0.75·R(proto) + 0.25·R(sed), α=1.5 )
#   + aggressive file-level amplification (top-3, power=0.50)
#   + harder noise gate, lower SED spike threshold
# ─────────────────────────────────────────────────────────────────────────
elif SUBMISSION_MODE == 3:
    p = M3
    print(f"Mode 3 | weights: proto={p["w_proto"]} sed={p["w_sed"]} | "
          f"power={p["power_alpha"]} | file_top_k={p["file_top_k"]} file_power={p["file_power"]}")

    # Step 1: heavy ProtoSSM-biased rank blend
    pred = p["w_proto"] * rank_proto + p["w_sed"] * rank_sed

    # Step 2: aggressive power sharpening — kills borderline predictions
    pred = power_sharpen(pred, p["power_alpha"])

    # Step 3: strong file-level rank amplification
    pred = file_rank_amplify(
        pred, file_ids,
        n_windows=N_WINDOWS,
        top_k=p["file_top_k"],
        file_power=p["file_power"],
    )
    pred = pred / (pred.max() + EPS)

    noise_thr_p = p["noise_thr_p"]
    noise_thr_s = p["noise_thr_s"]
    noise_pull  = p["noise_pull"]
    sed_spike_thr    = p["sed_spike_thr"]
    proto_spike_floor = p["proto_spike_floor"]


print(f"pred shape: {pred.shape} | min={pred.min():.5f} max={pred.max():.5f}")


## Cell 7 — Post-Processing Gates

In [ ]:
# Gate 1: Noise suppression
pred, fake_only = gate_noise_suppression(
    pred, rank_proto, p_proto, p_sed,
    thr_p=noise_thr_p, thr_s=noise_thr_s, pull=noise_pull
)
print(f"Gate 1 (noise suppression): {fake_only.sum():,} cells suppressed.")

# Gate 2: Temporal continuity
pred, proto_cont = gate_temporal_continuity(
    pred, rank_proto, rank_sed, p_proto, p_sed,
    file_ids, fake_only,
    ctx_pull=0.15,
)
print(f"Gate 2 (temporal continuity): {proto_cont.sum():,} cells boosted.")

# Gate 3: SED spike preservation
pred, sed_only = gate_sed_spike(
    pred, rank_proto, rank_sed,
    fake_only, proto_cont,
    sed_spike_thr=sed_spike_thr,
    proto_floor=proto_spike_floor,
    spike_pull=0.12,
)
print(f"Gate 3 (SED spike):          {sed_only.sum():,} cells preserved.")

# Final probability clipping to [0, 1]
pred = np.clip(pred, 0.0, 1.0).astype(np.float32)

# Build submission DataFrame
sub = df_proto[["row_id"]].copy()
sub[cols] = pred

print(f"Submission frame: {sub.shape} | dtype: {sub[cols].dtypes.unique()}")


## Cell 8 — Taxonomy Gates (Sonotype Mirror + Rare Class Suppression)

In [ ]:
BASE_PATH = Path("/kaggle/input/competitions/birdclef-2026")

# Gate 4: Sonotype mirroring
sub = gate_sonotype_mirror(sub, cols)

# Gate 5: Rare-class suppression
sub = gate_rare_class_suppress(
    sub, cols,
    taxonomy_path=BASE_PATH / "taxonomy.csv",
    suppress_factor=0.90,
    margin=0.05,
)

# ── Dry-run alignment ─────────────────────────────────────────────────────
# If no test soundscapes exist (local / notebook run), align to sample_submission
test_paths = list(BASE_PATH.glob("test_soundscapes/*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    print("Dry-run detected: aligning rows to sample_submission.csv")
    sample_public = pd.read_csv(BASE_PATH / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    sub = sample_public.copy()
    for label in cols:
        sub[label] = template[label]

sub.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  shape={sub.shape}")


## Cell 9 — Submission Diagnostics

In [ ]:
sub_check  = pd.read_csv(OUT_CSV)
prob_cols  = [c for c in sub_check.columns if c != "row_id"]
prob_vals  = sub_check[prob_cols].to_numpy(np.float32)

print("═" * 55)
print(f"  SUBMISSION_MODE     : {SUBMISSION_MODE}")
print(f"  Rows                : {len(sub_check):,}")
print(f"  Class columns       : {len(prob_cols)}")
print(f"  Missing values      : {int(sub_check.isna().sum().sum())}")
print(f"  Min probability     : {float(prob_vals.min()):.6f}")
print(f"  Max probability     : {float(prob_vals.max()):.6f}")
print(f"  Mean probability    : {float(prob_vals.mean()):.6f}")
print(f"  >0.5 cells          : {(prob_vals > 0.5).sum():,}")
print(f"  >0.9 cells          : {(prob_vals > 0.9).sum():,}")
print(f"  Duplicated row_id   : {int(sub_check["row_id"].duplicated().sum())}")
print("═" * 55)

# Hard assertions — any failure = do not submit
assert "row_id" in sub_check.columns,             "row_id column missing!"
assert len(prob_cols) > 0,                         "No probability columns found!"
assert np.isfinite(prob_vals).all(),               "Non-finite values in probabilities!"
assert float(prob_vals.min()) >= 0.0,              "Probability below 0!"
assert float(prob_vals.max()) <= 1.0 + 1e-5,      "Probability above 1!"
assert sub_check["row_id"].duplicated().sum() == 0, "Duplicate row_ids found!"

print(f"All checks passed. {OUT_CSV} is ready to submit.")


# References 
https://www.kaggle.com/code/nina2025/birdclef-2026-eos-4

https://www.kaggle.com/code/vyankteshdwivedi/birdclef-2026-protossm-sed-0-948

https://www.kaggle.com/code/raunakdey07/birdclef-2026-v6

https://www.kaggle.com/code/youssefmo942009/lb-0-948